# Notebook for creating QC plots for the Hi-C study

### Library loading

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import os
from itertools import chain
from itertools import combinations
import subprocess
from cytoolz import merge

# Hi-C utilities imports:
import cooler
import bioframe
import cooltools
from cooltools.api.saddle import saddle_strength
from cooltools.lib.numutils import fill_diag
import cooltools.lib.plotting
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.2'):
    raise AssertionError("tutorials rely on cooltools version 0.5.2 or higher,"+
                         "please check your cooltools version and update to the latest")

# Visualization imports:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import colors
%matplotlib inline
plt.style.use('seaborn-v0_8-poster')
from multiprocessing import Pool
from matplotlib.colors import LogNorm
import matplotlib.patches as patches
from matplotlib.ticker import EngFormatter

# helper functions for plotting
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.2'):
    raise AssertionError("tutorial relies on cooltools version 0.5.2 or higher,"+
                         "please check your cooltools version and update to the latest")

# count cpus
num_cpus = os.getenv('SLURM_CPUS_PER_TASK')
if not num_cpus:
    num_cpus = os.cpu_count()
num_cpus = int(num_cpus)

# Use bioframe to fetch the genomic features from the UCSC.
hg38_chromsizes = bioframe.fetch_chromsizes('hg38')
hg38_cens = bioframe.fetch_centromeres('hg38')
hg38_arms = bioframe.make_chromarms(hg38_chromsizes, hg38_cens)
hg38_arms.chrom=hg38_arms.chrom.str.replace("chr", "")
hg38_arms.name=hg38_arms.name.str.replace("chr", "")

### P(s) plots per sample!

In [ ]:
def ps_plotter(sampID, resolution):
    
    #Data load-in and p(s) calculation
    mycooler = cooler.Cooler(f'<path>/FULL_SAMPLES/{sampID}/cool/{sampID}.{resolution}_balanced.cool')
    cvd = cooltools.expected_cis(mycooler, view_df=hg38_arms, smooth=True, aggregate_smoothed=True, smooth_sigma=0.1, nproc=num_cpus) #cvd == contacts-vs-distance, smooth_sigma can be reduced for higher resolution?
    cvd['balanced.avg.smoothed'].loc[cvd['dist'] < 2] = np.nan #Set to NA for adjacent loci?
    cvd['balanced.avg.smoothed.agg'].loc[cvd['dist'] < 2] = np.nan #Set to NA for adjacent loci?
    
    #Plotting, non-aggregated
    f, ax = plt.subplots(1,1)

    for region in hg38_arms['name']:
        ax.loglog(
            cvd['dist_bp'].loc[cvd['region1']==region],
            cvd['balanced.avg.smoothed'].loc[cvd['region1']==region],
        )
        ax.set(
            xlabel='separation, bp',
            ylabel='IC contact frequency')
        ax.set_aspect(1.0)
        ax.grid(lw=0.5)
        
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_psplot_nonagg.png', dpi=300, bbox_inches="tight")
    plt.close(f)

    #Plotting, aggregated
    f, ax = plt.subplots(1,1)

    for region in hg38_arms['name']:
        ax.loglog(
            cvd['dist_bp'].loc[cvd['region1']==region],
            cvd['balanced.avg.smoothed.agg'].loc[cvd['region1']==region],
        )
        ax.set(
            xlabel='separation, bp',
            ylabel='IC contact frequency')
        ax.set_aspect(1.0)
        ax.grid(lw=0.5)
    
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_psplot_agg.png', dpi=300, bbox_inches="tight")
    plt.close(f)
    
    #Plotting of smoothed P(s) curve and its derivative
    cvd_merged = cvd.drop_duplicates(subset=['dist'])[['dist_bp', 'balanced.avg.smoothed.agg']]
    der = np.gradient(np.log(cvd_merged['balanced.avg.smoothed.agg']), np.log(cvd_merged['dist_bp']))
    
    f, axs = plt.subplots(
    figsize=(6.5,13),
    nrows=2,
    gridspec_kw={'height_ratios':[6,2]},
    sharex=True)
    ax = axs[0]
    ax.loglog(
        cvd_merged['dist_bp'],
        cvd_merged['balanced.avg.smoothed.agg'],
        '-'
    )

    ax.set(
        ylabel='IC contact frequency',
        xlim=(1e3,1e8)
    )
    ax.set_aspect(1.0)
    ax.grid(lw=0.5)


    ax = axs[1]
    ax.semilogx(
        cvd_merged['dist_bp'],
        der,
        alpha=0.5
    )

    ax.set(
        xlabel='separation, bp',
        ylabel='slope')

    ax.grid(lw=0.5)
    
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_psplot_deriv.png', dpi=300, bbox_inches="tight")
    plt.close(f)    
    
    cvd.to_csv(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_psdata.tsv', sep="\t", index=False)

In [ ]:
# Run them at 10kb, 5kb, and 2kb, just cause, those are the resolutions we most commonly used in the paper.
samples = [SAMPLE_01,SAMPLE_02,SAMPLE_03,SAMPLE_04,SAMPLE_05,SAMPLE_06,SAMPLE_07,SAMPLE_08,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
resolutions = [10000, 5000, 2000]
for samp in samples:
    for res in resolutions:
        ps_plotter(samp, res)

In [ ]:
def ps_plotter_multi_sample_derivative(basepath, samp_list, resolution, hg38_arms, num_cpus=num_cpus):
    """
    Plots a two-panel figure showing P(s) and its derivative across multiple samples.
    
    :param basepath: Base directory path for data
    :param samp_list: List of sample IDs (strings)
    :param resolution: Resolution (integer, e.g., 10000)
    :param hg38_arms: Genomic arms for expected calculations
    :param num_cpus: Number of CPU cores to use (default=4)
    """
    # Dictionary to store data and colors
    ps_data = {}
    sample_colors = {}  # To ensure matching colors across both panels

    # Generate unique colors for each sample
    color_map = plt.cm.viridis(np.linspace(0, 1, len(samp_list)))

    for i, sampID in enumerate(samp_list):
        try:
            # Data load-in and P(s) calculation
            mycooler = cooler.Cooler(f'{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool')
            cvd = cooltools.expected_cis(mycooler, view_df=hg38_arms, smooth=True, aggregate_smoothed=True, smooth_sigma=0.1, nproc=num_cpus)

            # Ensure adjacent loci are NaN
            cvd['balanced.avg.smoothed.agg'].loc[cvd['dist'] < 2] = np.nan

            # Drop duplicate distances and store P(s) values
            cvd_merged = cvd.drop_duplicates(subset=['dist'])[['dist_bp', 'balanced.avg.smoothed.agg']]

            # Store data for second plot
            ps_data[sampID] = (cvd_merged, color_map[i])  # Assign unique color to each sample
            sample_colors[sampID] = color_map[i]  # Store for matching between panels

        except Exception as e:
            print(f"Error processing sample {sampID}: {e}")
            continue

    ### **Two-Panel Plot: P(s) & Its Derivative** ###
    f, axs = plt.subplots(
        figsize=(6.5, 10),  # Adjusted size
        nrows=2,
        gridspec_kw={'height_ratios': [6, 2], 'hspace': 0.05},  # Reduced space between panels
        sharex=True
    )

    # **Top Panel: P(s) plot**
    ax = axs[0]
    for sampID, (cvd_merged, color) in ps_data.items():
        ax.loglog(
            cvd_merged['dist_bp'],
            cvd_merged['balanced.avg.smoothed.agg'],
            label=sampID,
            color=color,
            alpha=0.8
        )

    ax.set(
        ylabel='IC Contact Frequency',
        xlim=(1e3, 1e8)
    )
    ax.set_aspect(1.0)
    ax.grid(lw=0.5)
    ax.legend(fontsize=6, loc='lower left')  # Adjusted position to avoid overlap

    # **Bottom Panel: P(s) derivative plot**
    ax = axs[1]
    for sampID, (cvd_merged, color) in ps_data.items():
        der = np.gradient(np.log(cvd_merged['balanced.avg.smoothed.agg']), np.log(cvd_merged['dist_bp']))
        ax.semilogx(
            cvd_merged['dist_bp'],
            der,
            label=sampID,  # Legend added to maintain color consistency
            color=color,  # Match color with the top panel
            alpha=0.8
        )

    ax.set(
        xlabel='Separation (bp)',
        ylabel='Slope'
    )
    ax.grid(lw=0.5)

    # Save the figure
    os.makedirs(f'{basepath}/QCFIGS/', exist_ok=True)
    f.savefig(f'{basepath}/QCFIGS/multi_sample_psplot_deriv_{resolution}.pdf', dpi=300, bbox_inches="tight")
    plt.close(f)


In [ ]:
#samples = [SAMPLE_01,SAMPLE_02,SAMPLE_03,SAMPLE_04,SAMPLE_05,SAMPLE_06,SAMPLE_07,SAMPLE_08,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
ps_plotter_multi_sample_derivative('<path>/FULL_SAMPLES/', samples, 5000, hg38_arms, num_cpus=num_cpus)

### Coverage and cis/trans proportion plots per-sample

In [ ]:
def coverage_plotter(sampID, resolution):
    
    #Get Mb size
    scaled_resolution = int(resolution / 1_000_000)
    
    #Data load-in and coverage calculation
    mycooler = cooler.Cooler(f'<path>/FULL_SAMPLES/{sampID}/cool/{sampID}.{resolution}_balanced.cool')
    cis_coverage, tot_coverage = cooltools.coverage(mycooler, nproc=num_cpus)
    
    #Prepare to subset to autosomes + X and Y:
    chromstarts = [mycooler.extent(chrom)[0] for chrom in mycooler.chromnames[:24]]
    chromends = [mycooler.extent(chrom)[1] for chrom in mycooler.chromnames[:24]]
    full_extent_start = chromstarts[0]
    full_extent_end = chromends[-1]
    
    # Plot!
    f, ax = plt.subplots(
    figsize=(15, 16),
    )

    norm = LogNorm(vmax=0.1)

    im = ax.matshow(
        mycooler.matrix()[full_extent_start:full_extent_end,full_extent_start:full_extent_end],
        norm=norm,
        cmap='fall'
    );
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(im, cax=cax, label='corrected frequencies');
    ax.set_title(f'Genome-Wide Coverage, Sample {sampID}, @ {scaled_resolution}Mb Resolution ', pad=10)
    ax.xaxis.set_visible(False)

    ax1 = divider.append_axes("bottom", size="25%", pad=0.1, sharex=ax)
    weights = mycooler.bins()[:full_extent_end]['weight'].values
    ax1.plot( cis_coverage, label='cis')
    ax1.plot( tot_coverage, label='total')
    ax1.set_xlim([0, len(mycooler.bins()[:full_extent_end])])
    ax1.set_ylabel('Coverage')
    ax1.legend()
    ax1.set_xticks([])

    ax2 = divider.append_axes("bottom", size="25%", pad=0.1, sharex=ax)
    ax2.plot( cis_coverage/ tot_coverage)
    ax2.set_xlim([0, len(mycooler.bins()[:full_extent_end])])
    ax2.set_ylabel('coverage ratio')
    
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_coverageplot.png', dpi=300, bbox_inches="tight")
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_coverageplot.pdf', dpi=300, bbox_inches="tight")
    plt.close(f)

In [ ]:
# Run them at 500kb and 1Mb, just cause, this is genome wide, so much more reasonable approach
#samples = [SAMPLE_01,SAMPLE_02,SAMPLE_03,SAMPLE_04,SAMPLE_05,SAMPLE_06,SAMPLE_07,SAMPLE_08,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
#resolutions = [1000000, 500000]
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
resolutions= [1000000] #Only interested in 1 Mb
for samp in samples:
    for res in resolutions:
        coverage_plotter(samp, res)

In [ ]:
def coverage_plotter_multi_panel(
    samp_list,
    resolution,
    basepath="<path>/FULL_SAMPLES",
    outdir="<path>/FULL_SAMPLES/QCFIGS",
    ncols=3,
    num_cpus=num_cpus,
):
    import os
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import LogNorm
    import cooler
    import cooltools

    cis_color = "#1f77b4"
    total_color = "#ff7f0e"
    ratio_color = cis_color

    n_samples = len(samp_list)
    nrows = math.ceil(n_samples / ncols)
    scaled_resolution = resolution / 1_000_000

    fig = plt.figure(figsize=(ncols * 4.4, nrows * 4.9))

    outer = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols,
        figure=fig,
        left=0.05,
        right=0.90,
        bottom=0.04,
        top=0.95,
        wspace=0.25,
        hspace=0.32
    )

    norm = LogNorm(vmax=0.1)
    cmap = "fall"

    im_for_cbar = None
    heatmap_axes = []

    for idx, sampID in enumerate(samp_list):
        row = idx // ncols
        col = idx % ncols
        sampID = str(sampID)

        subgs = outer[row, col].subgridspec(
            3, 1,
            height_ratios=[4.0, 1.1, 1.1],
            hspace=0.06
        )

        ax_map = fig.add_subplot(subgs[0])
        ax_cov = fig.add_subplot(subgs[1], sharex=ax_map)
        ax_ratio = fig.add_subplot(subgs[2], sharex=ax_map)

        clr = cooler.Cooler(f"{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool")
        cis_coverage, tot_coverage = cooltools.coverage(clr, nproc=num_cpus)

        chromstarts = [clr.extent(chrom)[0] for chrom in clr.chromnames[:24]]
        chromends = [clr.extent(chrom)[1] for chrom in clr.chromnames[:24]]
        full_extent_start = chromstarts[0]
        full_extent_end = chromends[-1]

        mat = clr.matrix()[full_extent_start:full_extent_end, full_extent_start:full_extent_end]
        n_bins = full_extent_end - full_extent_start

        # Heatmap: make x-span align with tracks below
        im = ax_map.imshow(
            mat,
            norm=norm,
            cmap=cmap,
            interpolation="nearest",
            origin="upper",
            aspect="auto",
            extent=[0, n_bins, n_bins, 0],
            rasterized=True
        )

        if im_for_cbar is None:
            im_for_cbar = im

        heatmap_axes.append(ax_map)

        ax_map.set_xlim(0, n_bins)
        ax_map.set_ylim(n_bins, 0)
        ax_map.set_title(f"{sampID}", fontsize=9, pad=2)
        ax_map.set_xticks([])
        ax_map.set_yticks([])

        # Coverage panel
        ax_cov.plot(cis_coverage, lw=0.7, color=cis_color, label="cis")
        ax_cov.plot(tot_coverage, lw=0.7, color=total_color, label="total")
        ax_cov.set_xlim(0, n_bins)
        ax_cov.set_ylabel("Cov", fontsize=7)
        ax_cov.set_xticks([])
        ax_cov.tick_params(axis="y", labelsize=6)
        ax_cov.spines["top"].set_visible(False)
        ax_cov.spines["right"].set_visible(False)

        # Ratio panel
        ratio = np.divide(
            cis_coverage,
            tot_coverage,
            out=np.full_like(cis_coverage, np.nan, dtype=float),
            where=(tot_coverage != 0)
        )

        ax_ratio.plot(ratio, lw=0.7, color=ratio_color)
        ax_ratio.set_xlim(0, n_bins)
        ax_ratio.set_ylabel("cis/tot", fontsize=7)
        ax_ratio.set_xticks([])
        ax_ratio.tick_params(axis="y", labelsize=6)
        ax_ratio.spines["top"].set_visible(False)
        ax_ratio.spines["right"].set_visible(False)

    # Hide unused cells
    for idx in range(n_samples, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        ax_empty = fig.add_subplot(outer[row, col])
        ax_empty.axis("off")

    # Shared colorbar, attached only to heatmaps
    cbar = fig.colorbar(
        im_for_cbar,
        ax=heatmap_axes,
        fraction=0.025,
        pad=0.01
    )
    cbar.set_label("Corrected frequencies", fontsize=9)
    cbar.ax.tick_params(labelsize=7)

    # Shared legend with explicit colors
    cis_handle = mlines.Line2D([], [], color=cis_color, lw=1.5, label="cis")
    total_handle = mlines.Line2D([], [], color=total_color, lw=1.5, label="total")

    fig.legend(
        handles=[cis_handle, total_handle],
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=8,
        bbox_to_anchor=(0.5, 0.975)
    )

    fig.suptitle(
        f"Genome-wide coverage profiles across samples @ {scaled_resolution:g} Mb",
        fontsize=12,
        y=0.992
    )

    os.makedirs(outdir, exist_ok=True)
    outfile = f"{outdir}/multi_sample_coverageplot_{resolution}.pdf"

    fig.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig)

    print(f"Saved: {outfile}")

In [ ]:
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]

coverage_plotter_multi_panel(
    samp_list=samples,
    resolution=1000000,
    ncols=5
)

In [ ]:
def coverage_plotter_multi_panel(
    samp_list,
    resolution,
    basepath="<path>/FULL_SAMPLES",
    outdir="<path>/FULL_SAMPLES/QCFIGS",
    ncols=3,
    num_cpus=num_cpus,
):
    import os
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import LogNorm
    from matplotlib.ticker import FuncFormatter, MaxNLocator
    import cooler
    import cooltools

    cis_color = "#1f77b4"
    total_color = "#ff7f0e"
    ratio_color = cis_color

    n_samples = len(samp_list)
    nrows = math.ceil(n_samples / ncols)
    scaled_resolution = resolution / 1_000_000

    # final colorbar column reserved explicitly
    fig = plt.figure(figsize=(ncols * 4.4 + 0.7, nrows * 4.8))

    outer = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols + 1,
        figure=fig,
        width_ratios=[1] * ncols + [0.08],
        left=0.05,
        right=0.98,
        bottom=0.04,
        top=0.94,
        wspace=0.28,
        hspace=0.34
    )

    norm = LogNorm(vmax=0.1)
    cmap = "fall"
    im_for_cbar = None

    def plain_int(x, pos):
        if not np.isfinite(x):
            return ""
        return f"{int(round(x))}"

    for idx, sampID in enumerate(samp_list):
        row = idx // ncols
        col = idx % ncols
        sampID = str(sampID)

        subgs = outer[row, col].subgridspec(
            3, 1,
            height_ratios=[4.0, 1.15, 1.15],
            hspace=0.06
        )

        ax_map = fig.add_subplot(subgs[0])
        ax_cov = fig.add_subplot(subgs[1], sharex=ax_map)
        ax_ratio = fig.add_subplot(subgs[2], sharex=ax_map)

        clr = cooler.Cooler(f"{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool")
        cis_coverage, tot_coverage = cooltools.coverage(clr, nproc=num_cpus)

        chromstarts = [clr.extent(chrom)[0] for chrom in clr.chromnames[:24]]
        chromends = [clr.extent(chrom)[1] for chrom in clr.chromnames[:24]]
        full_extent_start = chromstarts[0]
        full_extent_end = chromends[-1]

        # Explicitly subset the same genomic extent for all three panels
        cis_cov = np.asarray(cis_coverage[full_extent_start:full_extent_end], dtype=float)
        tot_cov = np.asarray(tot_coverage[full_extent_start:full_extent_end], dtype=float)

        mat = clr.matrix(balance=True)[full_extent_start:full_extent_end, full_extent_start:full_extent_end]
        n_bins = full_extent_end - full_extent_start
        x = np.arange(n_bins)

        # Pixel-centered extent so image and line tracks align exactly
        im = ax_map.imshow(
            mat,
            norm=norm,
            cmap=cmap,
            interpolation="nearest",
            origin="upper",
            aspect="auto",
            extent=(-0.5, n_bins - 0.5, n_bins - 0.5, -0.5),
            rasterized=True
        )

        if im_for_cbar is None:
            im_for_cbar = im

        ax_map.set_xlim(-0.5, n_bins - 0.5)
        ax_map.set_ylim(n_bins - 0.5, -0.5)
        ax_map.set_title(f"{sampID}", fontsize=9, pad=2)
        ax_map.set_xticks([])
        ax_map.set_yticks([])
        ax_map.margins(x=0, y=0)

        # Coverage panel
        ax_cov.plot(x, cis_cov, lw=0.7, color=cis_color, label="cis")
        ax_cov.plot(x, tot_cov, lw=0.7, color=total_color, label="total")
        ax_cov.set_xlim(-0.5, n_bins - 0.5)
        ax_cov.set_ylabel("Cov", fontsize=7)
        ax_cov.set_xticks([])
        ax_cov.tick_params(axis="y", labelsize=6, length=2)
        ax_cov.spines["top"].set_visible(False)
        ax_cov.spines["right"].set_visible(False)
        ax_cov.margins(x=0)

        # turn off scientific-notation offset text like "1e6"
        ax_cov.yaxis.set_major_locator(MaxNLocator(nbins=3))
        ax_cov.yaxis.set_major_formatter(FuncFormatter(plain_int))
        ax_cov.yaxis.offsetText.set_visible(False)

        # Ratio panel
        ratio = np.divide(
            cis_cov,
            tot_cov,
            out=np.full_like(cis_cov, np.nan, dtype=float),
            where=(tot_cov != 0)
        )

        ax_ratio.plot(x, ratio, lw=0.7, color=ratio_color)
        ax_ratio.set_xlim(-0.5, n_bins - 0.5)
        ax_ratio.set_ylabel("cis/tot", fontsize=7)
        ax_ratio.set_xticks([])
        ax_ratio.tick_params(axis="y", labelsize=6, length=2)
        ax_ratio.spines["top"].set_visible(False)
        ax_ratio.spines["right"].set_visible(False)
        ax_ratio.margins(x=0)

    # Hide unused sample cells
    for idx in range(n_samples, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        ax_empty = fig.add_subplot(outer[row, col])
        ax_empty.axis("off")

    # Dedicated colorbar axis, outside the panel grid
    cax = fig.add_subplot(outer[:, -1])
    cbar = fig.colorbar(im_for_cbar, cax=cax)
    cbar.set_label("Corrected frequencies", fontsize=9)
    cbar.ax.tick_params(labelsize=7)

    # Shared legend with correct colors
    cis_handle = mlines.Line2D([], [], color=cis_color, lw=1.5, label="cis")
    total_handle = mlines.Line2D([], [], color=total_color, lw=1.5, label="total")

    fig.legend(
        handles=[cis_handle, total_handle],
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=8,
        bbox_to_anchor=(0.5, 0.975)
    )

    fig.suptitle(
        f"Genome-wide coverage profiles across samples @ {scaled_resolution:g} Mb",
        fontsize=12,
        y=0.992
    )

    os.makedirs(outdir, exist_ok=True)
    outfile = f"{outdir}/multi_sample_coverageplot_{resolution}.pdf"

    fig.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig)

    print(f"Saved: {outfile}")

In [ ]:
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]

coverage_plotter_multi_panel(
    samp_list=samples,
    resolution=1000000,
    ncols=5
)

### Compartments and Saddleplots per-sample

In [ ]:
## fasta sequence is required for calculating binned profile of GC conent
if not os.path.isfile('./hg38.fa'):
    ## note downloading a ~1Gb file can take a minute
    subprocess.call('wget --progress=bar:force:noscroll https://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz', shell=True)
    subprocess.call('gunzip hg38.fa.gz', shell=True)

hg38_genome = bioframe.load_fasta('./hg38.fa')

hg38_genome_nochr = {
    chrom.replace('chr', ''): seq
    for chrom, seq in hg38_genome.items()
}

In [ ]:
def saddleplot(
    track,
    saddledata,
    n_bins,
    vrange=None,
    qrange=(0.0, 1.0),
    cmap="coolwarm",
    scale="log",
    vmin=0.5,
    vmax=2,
    color=None,
    title=None,
    xlabel=None,
    ylabel=None,
    clabel=None,
    fig=None,
    fig_kws=None,
    heatmap_kws=None,
    margin_kws=None,
    cbar_kws=None,
    subplot_spec=None,
):
    """
    Generate a saddle plot.
    Parameters
    ----------
    track : pd.DataFrame
        See cooltools.digitize() for details.
    saddledata : 2D array-like
        Saddle matrix produced by `make_saddle`. It will include 2 flanking
        rows/columns for outlier signal values, thus the shape should be
        `(n+2, n+2)`.
    cmap : str or matplotlib colormap
        Colormap to use for plotting the saddle heatmap
    scale : str
        Color scaling to use for plotting the saddle heatmap: log or linear
    vmin, vmax : float
        Value limits for coloring the saddle heatmap
    color : matplotlib color value
        Face color for margin bar plots
    fig : matplotlib Figure, optional
        Specified figure to plot on. A new figure is created if none is
        provided.
    fig_kws : dict, optional
        Passed on to `plt.Figure()`
    heatmap_kws : dict, optional
        Passed on to `ax.imshow()`
    margin_kws : dict, optional
        Passed on to `ax.bar()` and `ax.barh()`
    cbar_kws : dict, optional
        Passed on to `plt.colorbar()`
    subplot_spec : GridSpec object
        Specify a subregion of a figure to using a GridSpec.
    Returns
    -------
    Dictionary of axes objects.
    """

#     warnings.warn(
#         "Generating a saddleplot will be deprecated in future versions, "
#         + "please see https://github.com/open2c_examples for examples on how to plot saddles.",
#         DeprecationWarning,
#     )

    from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
    from matplotlib.colors import Normalize, LogNorm
    from matplotlib import ticker
    import matplotlib.pyplot as plt

    class MinOneMaxFormatter(ticker.LogFormatter):
        def set_locs(self, locs=None):
            self._sublabels = set([vmin % 10 * 10, vmax % 10, 1])

        def __call__(self, x, pos=None):
            if x not in [vmin, 1, vmax]:
                return ""
            else:
                return "{x:g}".format(x=x)

    track_value_col = track.columns[3]
    track_values = track[track_value_col].values

    digitized_track, binedges = cooltools.digitize(
        track, n_bins, vrange=vrange, qrange=qrange
    )
    x = digitized_track[digitized_track.columns[3]].values.astype(int).copy()
    x = x[(x > -1) & (x < len(binedges) + 1)]

    # Old version
    # hist = np.bincount(x, minlength=len(binedges) + 1)

    groupmean = track[track.columns[3]].groupby(digitized_track[digitized_track.columns[3]]).mean()

    if qrange is not None:
        lo, hi = qrange
        binedges = np.linspace(lo, hi, n_bins + 1)

    # Barplot of mean values and saddledata are flanked by outlier bins
    n = saddledata.shape[0]
    X, Y = np.meshgrid(binedges, binedges)
    C = saddledata
    if (n - n_bins) == 2:
        C = C[1:-1, 1:-1]
        groupmean = groupmean[1:-1]

    # Layout
    if subplot_spec is not None:
        GridSpec = partial(GridSpecFromSubplotSpec, subplot_spec=subplot_spec)
    grid = {}
    gs = GridSpec(
        nrows=3,
        ncols=3,
        width_ratios=[0.2, 1, 0.1],
        height_ratios=[0.2, 1, 0.1],
        wspace=0.05,
        hspace=0.05,
    )

    # Figure
    if fig is None:
        fig_kws_default = dict(figsize=(10, 10))
        fig_kws = merge(fig_kws_default, fig_kws if fig_kws is not None else {})
        fig = plt.figure(**fig_kws)

    # Heatmap
    if scale == "log":
        norm = LogNorm(vmin=vmin, vmax=vmax)
    elif scale == "linear":
        norm = Normalize(vmin=vmin, vmax=vmax)
    else:
        raise ValueError("Only linear and log color scaling is supported")

    grid["ax_heatmap"] = ax = plt.subplot(gs[4])
    heatmap_kws_default = dict(cmap="coolwarm", rasterized=True)
    heatmap_kws = merge(
        heatmap_kws_default, heatmap_kws if heatmap_kws is not None else {}
    )
    img = ax.pcolormesh(X, Y, C, norm=norm, **heatmap_kws)
    plt.gca().yaxis.set_visible(False)

    # Margins
    margin_kws_default = dict(edgecolor="k", facecolor=color, linewidth=1)
    margin_kws = merge(margin_kws_default, margin_kws if margin_kws is not None else {})
    # left margin hist
    grid["ax_margin_y"] = plt.subplot(gs[3], sharey=grid["ax_heatmap"])

    plt.barh(
        binedges, height=1/len(binedges), width=groupmean, align="edge", **margin_kws
    )

    plt.xlim(plt.xlim()[1], plt.xlim()[0])  # fliplr
    plt.ylim(hi, lo)
    plt.gca().spines["top"].set_visible(False)
    plt.gca().spines["bottom"].set_visible(False)
    plt.gca().spines["left"].set_visible(False)
    plt.gca().xaxis.set_visible(False)
    # top margin hist
    grid["ax_margin_x"] = plt.subplot(gs[1], sharex=grid["ax_heatmap"])

    plt.bar(
        binedges, width=1/len(binedges), height=groupmean, align="edge", **margin_kws
    )

    plt.xlim(lo, hi)
    # plt.ylim(plt.ylim())  # correct
    plt.gca().spines["top"].set_visible(False)
    plt.gca().spines["right"].set_visible(False)
    plt.gca().spines["left"].set_visible(False)
    plt.gca().xaxis.set_visible(False)
    plt.gca().yaxis.set_visible(False)

#     # Colorbar
    grid["ax_cbar"] = plt.subplot(gs[5])
    cbar_kws_default = dict(fraction=0.8, label=clabel or "")
    cbar_kws = merge(cbar_kws_default, cbar_kws if cbar_kws is not None else {})
    if scale == "linear" and vmin is not None and vmax is not None:
        grid["ax_cbar"] = cb = plt.colorbar(img, **cbar_kws)
        # cb.set_ticks(np.arange(vmin, vmax + 0.001, 0.5))
        # # do linspace between vmin and vmax of 5 segments and trunc to 1 decimal:
        decimal = 10
        nsegments = 5
        cd_ticks = np.trunc(np.linspace(vmin, vmax, nsegments) * decimal) / decimal
        cb.set_ticks(cd_ticks)
    else:
        print('cbar')

        cb = plt.colorbar(img, format=MinOneMaxFormatter(), cax=grid["ax_cbar"], **cbar_kws)
        cb.ax.yaxis.set_minor_formatter(MinOneMaxFormatter())

    # extra settings
    grid["ax_heatmap"].set_xlim(lo, hi)
    grid["ax_heatmap"].set_ylim(hi, lo)
    grid['ax_heatmap'].grid(False)
    if title is not None:
        grid["ax_margin_x"].set_title(title)
    if xlabel is not None:
        grid["ax_heatmap"].set_xlabel(xlabel)
    if ylabel is not None:
        grid["ax_margin_y"].set_ylabel(ylabel)

    return grid

In [ ]:
def saddle_plotter(sampID, resolution):
    
    #Get Mb size
    scaled_resolution = int(resolution / 1_000_000)
    
    #Data load-in and coverage calculation
    mycooler = cooler.Cooler(f'<path>/FULL_SAMPLES/{sampID}/cool/{sampID}.{resolution}_balanced.cool')
    
    #Prepare to subset to autosomes + X and Y:
    chromstarts = [mycooler.extent(chrom)[0] for chrom in mycooler.chromnames[:24]]
    chromends = [mycooler.extent(chrom)[1] for chrom in mycooler.chromnames[:24]]
    full_extent_start = chromstarts[0]
    full_extent_end = chromends[-1]
    view_df = pd.DataFrame({'chrom':mycooler.chromnames[:24],'start':0,'end':mycooler.chromsizes.values[:24],'name':mycooler.chromnames[:24]})
    
    #Compartments
    #Define the range of GC content used to directionalize the compartments correctly
    bins = mycooler.bins()[:full_extent_end]
    gc_cov = bioframe.frac_gc(bins[['chrom', 'start', 'end']], hg38_genome_nochr)
    
    #Calculate compartments
    cis_eigs = cooltools.eigs_cis(mycooler, gc_cov, view_df=view_df, n_eigs=3)
    eigenvector_track = cis_eigs[1][['chrom', 'start', 'end', 'E1']]
    
    #Plot compartments and eigenvector 1:
    f, ax = plt.subplots(
    figsize=(15, 10),
    )

    norm = LogNorm(vmax=0.1)

    im = ax.matshow(
        mycooler.matrix()[full_extent_start:full_extent_end,full_extent_start:full_extent_end],
        norm=norm,
        cmap='fall'
    );
    mat = mycooler.matrix(balance=True)[full_extent_start:full_extent_end, full_extent_start:full_extent_end]
    n = mat.shape[0]
    plt.axis([0,n,n,0]) #This arbitrarily made the plot only 500 x 500 bins, change to be from 0-n for full extent of what's specified

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(im, cax=cax, label='corrected frequencies');
    ax.set_title(f'Genome-Wide Compartments, Sample {sampID} @ {scaled_resolution}Mb Resolution', pad=10)
    ax.set_ylabel('Genome-wide Bins')
    ax.xaxis.set_visible(False)

    ax1 = divider.append_axes("bottom", size="20%", pad=0.25, sharex=ax)
    weights = mycooler.bins()[:full_extent_end]['weight'].values
    ax1.plot([0,n],[0,0],'k',lw=0.25)
    ax1.plot( eigenvector_track['E1'].values, label='E1')

    ax1.set_ylabel('E1')
    ax1.set_xticks([]);


    for i in np.where(np.diff( (cis_eigs[1]['E1']>0).astype(int)))[0]:
        ax.plot([0, n],[i,i],'k',lw=0.5)
        ax.plot([i,i],[0, n],'k',lw=0.5)
    
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_compartment_plot.pdf', dpi=300, bbox_inches="tight")
    plt.close(f)
    
    #Plot compartment for only chr1:
    chr1_start, chr1_end = mycooler.extent("1")
    chr1_eigenvector = eigenvector_track[eigenvector_track['chrom'] == '1']
    
    f, ax = plt.subplots(
    figsize=(15, 10),
    )

    norm = LogNorm(vmax=0.1)

    im = ax.matshow(
        mycooler.matrix()[chr1_start:chr1_end,chr1_start:chr1_end],
        norm=norm,
        cmap='fall'
    );
    mat = mycooler.matrix(balance=True)[chr1_start:chr1_end,chr1_start:chr1_end]
    n = mat.shape[0]
    plt.axis([0,n,n,0]) #This arbitrarily made the plot only 500 x 500 bins, change to be from 0-n for full extent of what's specified

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(im, cax=cax, label='corrected frequencies');
    ax.set_title(f'Chr1 Compartments, Sample {sampID} @ {scaled_resolution}Mb Resolution', pad=10)
    ax.set_ylabel('Chr1 Bins')
    ax.xaxis.set_visible(False)

    ax1 = divider.append_axes("bottom", size="20%", pad=0.25, sharex=ax)
    weights = mycooler.bins()[chr1_start:chr1_end]['weight'].values
    ax1.plot([0,n],[0,0],'k',lw=0.25)
    ax1.plot( chr1_eigenvector['E1'].values, label='E1')

    ax1.set_ylabel('E1')
    ax1.set_xticks([]);


    for i in np.where(np.diff((chr1_eigenvector['E1'] > 0).astype(int)))[0]:
        ax.plot([0, n], [i, i], 'k', lw=0.5)
        ax.plot([i, i], [0, n], 'k', lw=0.5)
    
    f.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_chr1_compartment_plot.pdf', dpi=300, bbox_inches="tight")
    plt.close(f)
    
    
    #Saddleplots
    cvd = cooltools.expected_cis(clr=mycooler,view_df=view_df, nproc=num_cpus)
    Q_LO = 0.025 # ignore 2.5% of genomic bins with the lowest E1 values
    Q_HI = 0.975 # ignore 2.5% of genomic bins with the highest E1 values
    N_GROUPS = 38 # divide remaining 95% of the genome into 38 equisized groups, 2.5% each
    
    interaction_sum, interaction_count =  cooltools.saddle(mycooler,cvd,eigenvector_track,'cis',n_bins=N_GROUPS,qrange=(Q_LO,Q_HI),view_df=view_df)
    
    saddleplot(eigenvector_track,interaction_sum/interaction_count,N_GROUPS,qrange=(Q_LO,Q_HI),title=f'Sample {sampID} @ {scaled_resolution}Mb Resolution, Saddleplot',cbar_kws={'label':'average observed/expected contact frequency'})
    fig = plt.gcf()
    fig.set_size_inches(8, 12)
    fig.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_saddle_plot.pdf', dpi=300, bbox_inches="tight")
    plt.close(fig)
    
    x = np.arange(N_GROUPS + 2)
    plt.step(x, saddle_strength(interaction_sum, interaction_count), where='pre')
    plt.xlabel('extent')
    plt.ylabel('(AA + BB) / (AB + BA)')
    plt.title(f'Sample {sampID} @ {scaled_resolution}Mb Resolution, Saddle Strength Profile')
    plt.axhline(0, c='grey', ls='--', lw=1) # Q: is there a reason this is 0 not 1?
    plt.xlim(0, len(x)//2); # Q: is this less intuitive than showing for all x, as it converges to no difference (i.e. 1)?
    fig2 = plt.gcf()
    fig2.set_size_inches(8, 8)
    fig2.savefig(f'<path>/FULL_SAMPLES/QCFIGS/{sampID}.{resolution}_saddle_strength_plot.pdf', dpi=300, bbox_inches="tight")
    plt.close(fig2)


In [ ]:
# Run them at 500kb and 1Mb, just cause, this is genome wide, so much more reasonable approach
samples = [SAMPLE_01,SAMPLE_02,SAMPLE_03,SAMPLE_04,SAMPLE_05,SAMPLE_06,SAMPLE_07,SAMPLE_08,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
#resolutions = [1000000, 500000]
resolutions = [1000000] #Only care about 1 Mb these days!
for samp in samples:
    for res in resolutions:
        saddle_plotter(samp, res)

In [ ]:
def saddle_plotter_multi_panel(
    samp_list,
    resolution,
    hg38_genome_nochr,
    basepath="<path>/FULL_SAMPLES",
    outdir="<path>/FULL_SAMPLES/QCFIGS",
    ncols=5,
    num_cpus=96,
    q_lo=0.025,
    q_hi=0.975,
    n_groups=38,
    chroms_to_use=24,
    cmap="coolwarm",
    scale="log",
    vmin=0.5,
    vmax=2.0,
    margin_color="#BDBDBD",
):
    """
    Compute saddle plots for multiple samples and save them as one multi-panel PDF.

    Each sample panel contains:
      - top margin barplot
      - left margin barplot
      - central saddle heatmap

    The heatmaps share one global colorbar.

    Parameters
    ----------
    samp_list : list
        Sample IDs
    resolution : int
        Cooler resolution, e.g. 1000000 or 100000
    hg38_genome_nochr : object
        Genome reference object used by bioframe.frac_gc()
    basepath : str
        Base path containing per-sample cool files
    outdir : str
        Output directory
    ncols : int
        Number of columns in multi-panel layout
    num_cpus : int
        CPUs for cooltools functions
    q_lo, q_hi : float
        Quantile range for saddle
    n_groups : int
        Number of saddle bins
    chroms_to_use : int
        Number of chromosomes to include (default 24 = autosomes + X + Y)
    cmap : str
        Heatmap colormap
    scale : str
        "log" or "linear"
    vmin, vmax : float
        Color scale limits
    margin_color : str
        Color for top/left barplots
    """
    import os
    import math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import LogNorm, Normalize
    import cooler
    import cooltools
    import bioframe

    def format_resolution(res):
        if res % 1_000_000 == 0:
            return f"{res // 1_000_000} Mb"
        elif res % 1_000 == 0:
            return f"{res // 1_000} kb"
        else:
            return f"{res} bp"

    def draw_saddle_panel(fig, subplot_spec, track, saddledata, n_bins, qrange, title=None):
        """
        Draw one saddle panel inside a provided subplot spec.
        """
        from matplotlib.gridspec import GridSpecFromSubplotSpec

        # digitize track to reproduce the same marginal summaries as your original function
        digitized_track, binedges = cooltools.digitize(
            track, n_bins, vrange=None, qrange=qrange
        )

        track_value_col = track.columns[3]
        dig_value_col = digitized_track.columns[3]

        groupmean = track[track_value_col].groupby(digitized_track[dig_value_col]).mean()
        groupmean = np.asarray(groupmean, dtype=float)

        if qrange is not None:
            lo, hi = qrange
            binedges = np.linspace(lo, hi, n_bins + 1)
        else:
            lo, hi = binedges[0], binedges[-1]

        C = np.asarray(saddledata, dtype=float)

        # remove flanking outlier bins if present
        if (C.shape[0] - n_bins) == 2:
            C = C[1:-1, 1:-1]
            if len(groupmean) >= n_bins + 2:
                groupmean = groupmean[1:-1]

        # safety in case groupmean length is slightly off
        if len(groupmean) > n_bins:
            groupmean = groupmean[:n_bins]
        elif len(groupmean) < n_bins:
            groupmean = np.pad(groupmean, (0, n_bins - len(groupmean)), constant_values=np.nan)

        # mesh for pcolormesh
        X, Y = np.meshgrid(binedges, binedges)

        # per-sample layout: top margin, left margin, heatmap
        gs = GridSpecFromSubplotSpec(
            nrows=2,
            ncols=2,
            subplot_spec=subplot_spec,
            width_ratios=[0.24, 1.0],
            height_ratios=[0.24, 1.0],
            wspace=0.03,
            hspace=0.03,
        )

        ax_top = fig.add_subplot(gs[0, 1])
        ax_left = fig.add_subplot(gs[1, 0])
        ax_heat = fig.add_subplot(gs[1, 1], sharex=ax_top, sharey=ax_left)

        # norm
        if scale == "log":
            norm = LogNorm(vmin=vmin, vmax=vmax)
        elif scale == "linear":
            norm = Normalize(vmin=vmin, vmax=vmax)
        else:
            raise ValueError("scale must be 'log' or 'linear'")

        # heatmap
        img = ax_heat.pcolormesh(
            X, Y, C,
            cmap=cmap,
            norm=norm,
            shading="auto",
            rasterized=True
        )
        ax_heat.set_xlim(lo, hi)
        ax_heat.set_ylim(hi, lo)
        ax_heat.grid(False)
        ax_heat.set_xticks([])
        ax_heat.set_yticks([])

        # left margin
        ax_left.barh(
            binedges[:-1],
            groupmean,
            height=np.diff(binedges),
            align="edge",
            facecolor=margin_color,
            edgecolor="k",
            linewidth=0.5
        )
        ax_left.set_ylim(hi, lo)
        ax_left.invert_xaxis()
        ax_left.set_xticks([])
        ax_left.set_yticks([])
        ax_left.spines["top"].set_visible(False)
        ax_left.spines["bottom"].set_visible(False)
        ax_left.spines["left"].set_visible(False)

        # top margin
        ax_top.bar(
            binedges[:-1],
            groupmean,
            width=np.diff(binedges),
            align="edge",
            facecolor=margin_color,
            edgecolor="k",
            linewidth=0.5
        )
        ax_top.set_xlim(lo, hi)
        ax_top.set_xticks([])
        ax_top.set_yticks([])
        ax_top.spines["top"].set_visible(False)
        ax_top.spines["right"].set_visible(False)
        ax_top.spines["left"].set_visible(False)

        if title is not None:
            ax_top.set_title(title, fontsize=9, pad=2)

        return img

    # ---------------------------------------------------------
    # compute all sample saddles
    # ---------------------------------------------------------
    results = []

    for sampID in samp_list:
        try:
            sampID = str(sampID)

            clr = cooler.Cooler(f"{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool")

            view_df = pd.DataFrame({
                "chrom": clr.chromnames[:chroms_to_use],
                "start": 0,
                "end": clr.chromsizes.values[:chroms_to_use],
                "name": clr.chromnames[:chroms_to_use]
            })

            chromstarts = [clr.extent(chrom)[0] for chrom in clr.chromnames[:chroms_to_use]]
            chromends = [clr.extent(chrom)[1] for chrom in clr.chromnames[:chroms_to_use]]
            full_extent_start = chromstarts[0]
            full_extent_end = chromends[-1]

            bins = clr.bins()[:full_extent_end]
            gc_cov = bioframe.frac_gc(
                bins[["chrom", "start", "end"]],
                hg38_genome_nochr
            )

            cis_eigs = cooltools.eigs_cis(
                clr,
                gc_cov,
                view_df=view_df,
                n_eigs=3
            )

            eigenvector_track = cis_eigs[1][["chrom", "start", "end", "E1"]]

            cvd = cooltools.expected_cis(
                clr=clr,
                view_df=view_df,
                nproc=num_cpus
            )

            interaction_sum, interaction_count = cooltools.saddle(
                clr,
                cvd,
                eigenvector_track,
                "cis",
                n_bins=n_groups,
                qrange=(q_lo, q_hi),
                view_df=view_df
            )

            saddle_matrix = interaction_sum / interaction_count

            results.append({
                "sample": sampID,
                "track": eigenvector_track,
                "saddle_matrix": saddle_matrix
            })

            print(f"Processed sample {sampID}")

        except Exception as e:
            print(f"Error processing sample {sampID}: {e}")
            continue

    if len(results) == 0:
        raise RuntimeError("No saddle plots were successfully computed.")

    # ---------------------------------------------------------
    # plot multi-panel figure
    # ---------------------------------------------------------
    n_samples = len(results)
    nrows = math.ceil(n_samples / ncols)
    res_label = format_resolution(resolution)

    fig = plt.figure(figsize=(ncols * 4.0 + 0.7, nrows * 4.0))

    outer = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols + 1,
        figure=fig,
        width_ratios=[1] * ncols + [0.08],
        left=0.05,
        right=0.98,
        bottom=0.04,
        top=0.94,
        wspace=0.22,
        hspace=0.28
    )

    im_for_cbar = None

    for idx, res in enumerate(results):
        row = idx // ncols
        col = idx % ncols

        im = draw_saddle_panel(
            fig=fig,
            subplot_spec=outer[row, col],
            track=res["track"],
            saddledata=res["saddle_matrix"],
            n_bins=n_groups,
            qrange=(q_lo, q_hi),
            title=res["sample"]
        )

        if im_for_cbar is None:
            im_for_cbar = im

    # hide unused cells
    for idx in range(n_samples, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        ax_empty = fig.add_subplot(outer[row, col])
        ax_empty.axis("off")

    # shared colorbar
    cax = fig.add_subplot(outer[:, -1])
    cbar = fig.colorbar(im_for_cbar, cax=cax)

    if scale == "log":
        cbar.set_ticks([vmin, 1, vmax])
        cbar.set_ticklabels([f"{vmin:g}", "1", f"{vmax:g}"])

    cbar.set_label("Average observed/expected contact frequency", fontsize=9)
    cbar.ax.tick_params(labelsize=7)

    # figure-level labels/title
    fig.supxlabel("Compartment score quantile", fontsize=10, y=0.015)
    fig.supylabel("Compartment score quantile", fontsize=10, x=0.01)
    fig.suptitle(f"Multi-sample saddle plots @ {res_label}", fontsize=12, y=0.985)

    os.makedirs(outdir, exist_ok=True)
    outfile = f"{outdir}/multi_sample_saddle_plot_{resolution}.pdf"

    fig.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig)

    print(f"Saved: {outfile}")

In [ ]:
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
saddle_plotter_multi_panel(samp_list=samples, resolution=100000, hg38_genome_nochr=hg38_genome_nochr)

In [ ]:
def saddle_plotter_multi_panel(
    samp_list,
    resolution,
    hg38_genome_nochr,
    basepath="<path>/FULL_SAMPLES",
    outdir="<path>/FULL_SAMPLES/QCFIGS",
    ncols=5,
    num_cpus=96,
    q_lo=0.025,
    q_hi=0.975,
    n_groups=38,
    chroms_to_use=24,
    cmap="coolwarm",
    scale="log",
    vmin=0.5,
    vmax=2.0,
    margin_color="#2C7FB8",
):
    import os
    import math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import LogNorm, Normalize
    from matplotlib.ticker import FixedLocator, FixedFormatter
    import cooler
    import cooltools
    import bioframe

    def format_resolution(res):
        if res % 1_000_000 == 0:
            return f"{res // 1_000_000} Mb"
        elif res % 1_000 == 0:
            return f"{res // 1_000} kb"
        else:
            return f"{res} bp"

    def draw_saddle_panel(fig, subplot_spec, track, saddledata, n_bins, qrange, title=None):
        from matplotlib.gridspec import GridSpecFromSubplotSpec

        digitized_track, binedges = cooltools.digitize(
            track, n_bins, vrange=None, qrange=qrange
        )

        track_value_col = track.columns[3]
        dig_value_col = digitized_track.columns[3]

        groupmean = track[track_value_col].groupby(digitized_track[dig_value_col]).mean()
        groupmean = np.asarray(groupmean, dtype=float)

        if qrange is not None:
            lo, hi = qrange
            binedges = np.linspace(lo, hi, n_bins + 1)
        else:
            lo, hi = binedges[0], binedges[-1]

        C = np.asarray(saddledata, dtype=float)

        # remove flanking outlier bins if present
        if (C.shape[0] - n_bins) == 2:
            C = C[1:-1, 1:-1]
            if len(groupmean) >= n_bins + 2:
                groupmean = groupmean[1:-1]

        if len(groupmean) > n_bins:
            groupmean = groupmean[:n_bins]
        elif len(groupmean) < n_bins:
            groupmean = np.pad(groupmean, (0, n_bins - len(groupmean)), constant_values=np.nan)

        X, Y = np.meshgrid(binedges, binedges)

        gs = GridSpecFromSubplotSpec(
            nrows=2,
            ncols=2,
            subplot_spec=subplot_spec,
            width_ratios=[0.24, 1.0],
            height_ratios=[0.24, 1.0],
            wspace=0.03,
            hspace=0.03,
        )

        # IMPORTANT: do not share axes here
        ax_top = fig.add_subplot(gs[0, 1])
        ax_left = fig.add_subplot(gs[1, 0])
        ax_heat = fig.add_subplot(gs[1, 1])

        if scale == "log":
            norm = LogNorm(vmin=vmin, vmax=vmax)
        elif scale == "linear":
            norm = Normalize(vmin=vmin, vmax=vmax)
        else:
            raise ValueError("scale must be 'log' or 'linear'")

        img = ax_heat.pcolormesh(
            X, Y, C,
            cmap=cmap,
            norm=norm,
            shading="auto",
            rasterized=True
        )
        ax_heat.set_xlim(lo, hi)
        ax_heat.set_ylim(hi, lo)
        ax_heat.grid(False)

        # left margin
        ax_left.barh(
            binedges[:-1],
            groupmean,
            height=np.diff(binedges),
            align="edge",
            facecolor=margin_color,
            edgecolor="k",
            linewidth=0.35
        )
        ax_left.set_ylim(hi, lo)
        ax_left.invert_xaxis()
        ax_left.set_xticks([])
        ax_left.spines["top"].set_visible(False)
        ax_left.spines["bottom"].set_visible(False)
        ax_left.spines["left"].set_visible(False)

        # top margin
        ax_top.bar(
            binedges[:-1],
            groupmean,
            width=np.diff(binedges),
            align="edge",
            facecolor=margin_color,
            edgecolor="k",
            linewidth=0.35
        )
        ax_top.set_xlim(lo, hi)
        ax_top.set_xticks([])
        ax_top.set_yticks([])
        ax_top.spines["top"].set_visible(False)
        ax_top.spines["right"].set_visible(False)
        ax_top.spines["left"].set_visible(False)

        # NOW set heatmap ticks, after all limits are final
        tick_vals = [0.2, 0.4, 0.6, 0.8]

        # x ticks stay on the heatmap
        ax_heat.set_xticks(tick_vals)
        ax_heat.set_xticklabels([str(x) for x in tick_vals], fontsize=7)
        ax_heat.tick_params(axis="x", labelsize=7, length=2)

        # y ticks move to the left margin axis, like in the docs
        ax_heat.set_yticks([])
        ax_heat.yaxis.set_visible(False)

        ax_left.set_yticks(tick_vals)
        ax_left.set_yticklabels([str(x) for x in tick_vals], fontsize=7)
        ax_left.tick_params(axis="y", labelsize=7, length=2, pad=2)

        for lab in ax_heat.get_yticklabels():
            lab.set_horizontalalignment("right")

        if title is not None:
            ax_top.set_title(title, fontsize=9, pad=2)

        return img

    results = []

    for sampID in samp_list:
        try:
            sampID = str(sampID)

            clr = cooler.Cooler(f"{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool")

            view_df = pd.DataFrame({
                "chrom": clr.chromnames[:chroms_to_use],
                "start": 0,
                "end": clr.chromsizes.values[:chroms_to_use],
                "name": clr.chromnames[:chroms_to_use]
            })

            chromends = [clr.extent(chrom)[1] for chrom in clr.chromnames[:chroms_to_use]]
            full_extent_end = chromends[-1]

            bins = clr.bins()[:full_extent_end]
            gc_cov = bioframe.frac_gc(
                bins[["chrom", "start", "end"]],
                hg38_genome_nochr
            )

            cis_eigs = cooltools.eigs_cis(
                clr,
                gc_cov,
                view_df=view_df,
                n_eigs=3
            )

            eigenvector_track = cis_eigs[1][["chrom", "start", "end", "E1"]]

            cvd = cooltools.expected_cis(
                clr=clr,
                view_df=view_df,
                nproc=num_cpus
            )

            interaction_sum, interaction_count = cooltools.saddle(
                clr,
                cvd,
                eigenvector_track,
                "cis",
                n_bins=n_groups,
                qrange=(q_lo, q_hi),
                view_df=view_df
            )

            saddle_matrix = interaction_sum / interaction_count

            results.append({
                "sample": sampID,
                "track": eigenvector_track,
                "saddle_matrix": saddle_matrix
            })

            print(f"Processed sample {sampID}")

        except Exception as e:
            print(f"Error processing sample {sampID}: {e}")
            continue

    if len(results) == 0:
        raise RuntimeError("No saddle plots were successfully computed.")

    n_samples = len(results)
    nrows = math.ceil(n_samples / ncols)
    res_label = format_resolution(resolution)

    fig = plt.figure(figsize=(ncols * 4.0 + 0.8, nrows * 4.0))

    outer = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols + 1,
        figure=fig,
        width_ratios=[1] * ncols + [0.10],
        left=0.05,
        right=0.98,
        bottom=0.05,
        top=0.94,
        wspace=0.22,
        hspace=0.28
    )

    im_for_cbar = None

    for idx, res in enumerate(results):
        row = idx // ncols
        col = idx % ncols

        im = draw_saddle_panel(
            fig=fig,
            subplot_spec=outer[row, col],
            track=res["track"],
            saddledata=res["saddle_matrix"],
            n_bins=n_groups,
            qrange=(q_lo, q_hi),
            title=res["sample"]
        )

        if im_for_cbar is None:
            im_for_cbar = im

    for idx in range(n_samples, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        ax_empty = fig.add_subplot(outer[row, col])
        ax_empty.axis("off")

    cax = fig.add_subplot(outer[:, -1])
    cbar = fig.colorbar(im_for_cbar, cax=cax)

    # Force simple major ticks and remove the big scientific log annotation
    cbar.set_ticks([vmin, 1, vmax])
    cbar.set_ticklabels([f"{vmin:g}", "1", f"{vmax:g}"])
    cbar.ax.yaxis.set_minor_locator(FixedLocator([]))
    cbar.minorticks_off()
    cbar.ax.yaxis.get_offset_text().set_visible(False)

    cbar.set_label("Average observed/expected contact frequency", fontsize=14)
    cbar.ax.tick_params(labelsize=10)

    fig.supxlabel("Compartment Score Quantile", fontsize=11, y=0.015)
    fig.supylabel("Compartment Score Quantile", fontsize=11, x=0.01)
    fig.suptitle(f"Saddle Plots @ {res_label}", fontsize=13, y=0.985)

    os.makedirs(outdir, exist_ok=True)
    outfile = f"{outdir}/multi_sample_saddle_plot_{res_label}.pdf"

    fig.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig)

    print(f"Saved: {outfile}")

In [ ]:
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
#samples = [SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22] #smaller test
saddle_plotter_multi_panel(samp_list=samples, resolution=100000, hg38_genome_nochr=hg38_genome_nochr)

In [ ]:
def saddle_strength_plotter_multi_sample_fixed(
    basepath,
    samp_list,
    resolution,
    hg38_genome_nochr,
    num_cpus=num_cpus,
    q_lo=0.025,
    q_hi=0.975,
    n_groups=38,
    chroms_to_use=24,
    xmax_half=True
):
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import cooler
    import cooltools
    import bioframe

    saddle_data = {}
    color_map = plt.cm.viridis(np.linspace(0, 1, len(samp_list)))

    for i, sampID in enumerate(samp_list):
        try:
            sampID = str(sampID)

            clr = cooler.Cooler(
                f"{basepath}/{sampID}/cool/{sampID}.{resolution}_balanced.cool"
            )

            view_df = pd.DataFrame({
                "chrom": clr.chromnames[:chroms_to_use],
                "start": 0,
                "end": clr.chromsizes.values[:chroms_to_use],
                "name": clr.chromnames[:chroms_to_use]
            })

            full_extent_end = clr.extent(clr.chromnames[:chroms_to_use][-1])[1]
            bins = clr.bins()[:full_extent_end]

            gc_cov = bioframe.frac_gc(
                bins[["chrom", "start", "end"]],
                hg38_genome_nochr
            )

            cis_eigs = cooltools.eigs_cis(
                clr,
                gc_cov,
                view_df=view_df,
                n_eigs=3
            )
            eigenvector_track = cis_eigs[1][["chrom", "start", "end", "E1"]]

            cvd = cooltools.expected_cis(
                clr=clr,
                view_df=view_df,
                nproc=num_cpus
            )

            interaction_sum, interaction_count = cooltools.saddle(
                clr,
                cvd,
                eigenvector_track,
                "cis",
                n_bins=n_groups,
                qrange=(q_lo, q_hi),
                view_df=view_df
            )

            strength = np.asarray(saddle_strength(interaction_sum, interaction_count), dtype=float)
            if len(strength) < 2:
                continue
            
            S = interaction_sum / interaction_count
            strength[0] = (S[0,0] + S[-1,-1]) / (S[-1,0] + S[0,-1])

            x = np.arange(len(strength))

            saddle_data[sampID] = (x, strength, color_map[i])

        except Exception as e:
            print(f"Error processing sample {sampID}: {e}")
            continue

    fig, ax = plt.subplots(figsize=(7, 5.5))

    for sampID, (x, strength, color) in saddle_data.items():
        ax.step(
            x,
            strength,
            where="pre",
            label=sampID,
            color=color,
            alpha=0.85
        )

    ax.set_xlabel("Extent")
    ax.set_ylabel("(AA + BB) / (AB + BA)")
    ax.set_title(f"Saddle Strength Profiles @ {resolution // 1_000:,} kb")
    ax.axhline(1, c="grey", ls="--", lw=1)
    ax.grid(lw=0.5)
    ax.legend(fontsize=6, loc="best", ncol=2)

    if len(saddle_data) > 0:
        any_x = next(iter(saddle_data.values()))[0]
        if xmax_half:
            ax.set_xlim(0, len(any_x) // 2)
        else:
            ax.set_xlim(0, len(any_x) - 1)

    os.makedirs(f"{basepath}/QCFIGS/", exist_ok=True)
    fig.savefig(
        f"{basepath}/QCFIGS/multi_sample_saddle_strength_{resolution // 1_000}kb.pdf",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

In [ ]:
samples = [SAMPLE_05,SAMPLE_09,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22]
saddle_strength_plotter_multi_sample_fixed("<path>/FULL_SAMPLES/", samples, 100000, hg38_genome_nochr)